# Notebook 11 — Construcción del dataset final de modelado

Este notebook integra en una única tabla trimestral todas las capas construidas en los notebooks anteriores del pipeline:

- el **target trimestral de clasificación**,
- las **variables meteorológicas trimestrales**,
- las **variables hidrológicas trimestrales**,
- y las **variables trimestrales de presión atmosférica**.

El objetivo es dejar preparado el **dataset final de modelado**, con una fila por trimestre y con todas las variables explicativas ya alineadas temporalmente.

## Objetivo

- Cargar los datasets trimestrales generados en los notebooks 07, 08, 09 y 10.
- Validar shapes, claves y posibles duplicados.
- Integrar todas las fuentes mediante merges por `year` y `year_quarter`.
- Revisar valores nulos y consistencia del resultado.
- Seleccionar y guardar la versión final del dataset para el notebook de modelado.

## Preparación del entorno

Montaje de Google Drive e importación de las librerías necesarias para cargar, validar e integrar las distintas tablas trimestrales del pipeline.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

## Rutas de entrada y salida

Se definen las rutas de los cuatro bloques que se van a integrar y la ruta de salida del dataset final de modelado.

In [3]:
BASE_DIR = Path("/content/drive/MyDrive/PIDS4jjj2")
OUT_DIR = BASE_DIR / "outputs" / "llm_activity"

TARGET_PATH = OUT_DIR / "valmayor_target_quarter_classification.parquet"
METEO_PATH = OUT_DIR / "valmayor_target_meteo_quarter.parquet"
HYDRO_PATH = OUT_DIR / "valmayor_target_hydro_quarter.parquet"
PRESSURE_PATH = OUT_DIR / "valmayor_target_pressure_quarter.parquet"

FINAL_DATASET_PATH = OUT_DIR / "valmayor_modeling_dataset.parquet"
FINAL_DATASET_CSV_PATH = OUT_DIR / "valmayor_modeling_dataset.csv"

print("TARGET_PATH:", TARGET_PATH)
print("METEO_PATH:", METEO_PATH)
print("HYDRO_PATH:", HYDRO_PATH)
print("PRESSURE_PATH:", PRESSURE_PATH)
print("FINAL_DATASET_PATH:", FINAL_DATASET_PATH)

TARGET_PATH: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity/valmayor_target_quarter_classification.parquet
METEO_PATH: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity/valmayor_target_meteo_quarter.parquet
HYDRO_PATH: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity/valmayor_target_hydro_quarter.parquet
PRESSURE_PATH: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity/valmayor_target_pressure_quarter.parquet
FINAL_DATASET_PATH: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity/valmayor_modeling_dataset.parquet


## Carga de los bloques trimestrales

Se cargan las cuatro tablas trimestrales que se van a combinar en el dataset final.

In [4]:
df_target = pd.read_parquet(TARGET_PATH)
df_meteo = pd.read_parquet(METEO_PATH)
df_hydro = pd.read_parquet(HYDRO_PATH)
df_pressure = pd.read_parquet(PRESSURE_PATH)

print("Shape target:", df_target.shape)
print("Shape meteo:", df_meteo.shape)
print("Shape hydro:", df_hydro.shape)
print("Shape pressure:", df_pressure.shape)

Shape target: (54, 11)
Shape meteo: (54, 21)
Shape hydro: (54, 21)
Shape pressure: (54, 21)


In [5]:
df_target.head(3)

,year,year_quarter,n_videos,mean_activity_mentions,share_low,share_medium,share_high,share_certainty_high,captures_nonnull_mean,target_class,target_class_id
0,2009.0,2009-Q3,2,1.0,1.0,0.0,0.0,0.0,NaN,low,0
1,2009.0,2009-Q4,1,1.0,0.0,1.0,0.0,1.0,1.0,medium,1
2,2010.0,2010-Q1,1,1.0,1.0,0.0,0.0,0.0,NaN,low,0


## Comprobación de claves de integración

Antes de fusionar las tablas, se revisa que todas contengan las claves temporales necesarias: `year` y `year_quarter`.

In [6]:
for name, df in {
    "target": df_target,
    "meteo": df_meteo,
    "hydro": df_hydro,
    "pressure": df_pressure
}.items():
    print("=" * 80)
    print(name)
    print("Columnas clave presentes:", "year" in df.columns, "year_quarter" in df.columns)
    print(df[["year", "year_quarter"]].head(3))

target
Columnas clave presentes: True True
     year year_quarter
0  2009.0      2009-Q3
1  2009.0      2009-Q4
2  2010.0      2010-Q1
meteo
Columnas clave presentes: True True
     year year_quarter
0  2009.0      2009-Q3
1  2009.0      2009-Q4
2  2010.0      2010-Q1
hydro
Columnas clave presentes: True True
     year year_quarter
0  2009.0      2009-Q3
1  2009.0      2009-Q4
2  2010.0      2010-Q1
pressure
Columnas clave presentes: True True
     year year_quarter
0  2009.0      2009-Q3
1  2009.0      2009-Q4
2  2010.0      2010-Q1


## Revisión de duplicados por trimestre

Se comprueba que cada bloque tenga como máximo una fila por `year_quarter`, ya que el dataset final debe conservar una única fila por trimestre.

In [7]:
for name, df in {
    "target": df_target,
    "meteo": df_meteo,
    "hydro": df_hydro,
    "pressure": df_pressure
}.items():
    n_dups = df.duplicated(subset=["year_quarter"]).sum()
    print(f"{name}: duplicados por year_quarter = {n_dups}")

target: duplicados por year_quarter = 0
meteo: duplicados por year_quarter = 0
hydro: duplicados por year_quarter = 0
pressure: duplicados por year_quarter = 0


## Selección de columnas útiles por bloque

Como los notebooks 08, 09 y 10 ya contienen internamente el target tras el merge, aquí se seleccionan únicamente las columnas nuevas de cada bloque para evitar repetir variables.

In [8]:
base_cols = [
    "year",
    "year_quarter",
    "n_videos",
    "mean_activity_mentions",
    "share_low",
    "share_medium",
    "share_high",
    "share_certainty_high",
    "captures_nonnull_mean",
    "target_class",
    "target_class_id"
]

df_target_base = df_target[base_cols].copy()

meteo_feature_cols = [
    c for c in df_meteo.columns
    if c not in df_target.columns
]

hydro_feature_cols = [
    c for c in df_hydro.columns
    if c not in df_target.columns
]

pressure_feature_cols = [
    c for c in df_pressure.columns
    if c not in df_target.columns
]

print("Nuevas columnas meteo:", meteo_feature_cols)
print("Nuevas columnas hydro:", hydro_feature_cols)
print("Nuevas columnas pressure:", pressure_feature_cols)

Nuevas columnas meteo: ['quarter', 'temp_mean_quarter', 'temp_max_mean_quarter', 'temp_min_mean_quarter', 'precip_sum_quarter', 'rain_sum_quarter', 'snowfall_sum_quarter', 'wind_max_mean_quarter', 'radiation_sum_quarter', 'eto_sum_quarter']
Nuevas columnas hydro: ['quarter', 'agua_total_mean_quarter', 'agua_total_min_quarter', 'agua_total_max_quarter', 'agua_total_std_quarter', 'agua_actual_mean_quarter', 'agua_actual_min_quarter', 'agua_actual_max_quarter', 'agua_actual_std_quarter', 'n_obs_embalse_quarter']
Nuevas columnas pressure: ['quarter', 'pressure_msl_mean_quarter', 'pressure_msl_min_quarter', 'pressure_msl_max_quarter', 'pressure_msl_std_quarter', 'surface_pressure_mean_quarter', 'surface_pressure_min_quarter', 'surface_pressure_max_quarter', 'surface_pressure_std_quarter', 'n_obs_pressure_quarter']


In [9]:
df_meteo_feat = df_meteo[["year", "year_quarter"] + meteo_feature_cols].copy()
df_hydro_feat = df_hydro[["year", "year_quarter"] + hydro_feature_cols].copy()
df_pressure_feat = df_pressure[["year", "year_quarter"] + pressure_feature_cols].copy()

print(df_meteo_feat.shape)
print(df_hydro_feat.shape)
print(df_pressure_feat.shape)

(54, 12)
(54, 12)
(54, 12)


## Integración de las fuentes

Se construye el dataset final mediante merges sucesivos por `year` y `year_quarter`, tomando el target trimestral como base principal.

In [10]:
df_model = (
    df_target_base
    .merge(df_meteo_feat, on=["year", "year_quarter"], how="left")
    .merge(df_hydro_feat, on=["year", "year_quarter"], how="left")
    .merge(df_pressure_feat, on=["year", "year_quarter"], how="left")
    .sort_values(["year", "year_quarter"])
    .reset_index(drop=True)
)

print("Shape final tras merges:", df_model.shape)
df_model.head(10)

Shape final tras merges: (54, 41)


,year,year_quarter,n_videos,mean_activity_mentions,share_low,share_medium,share_high,share_certainty_high,captures_nonnull_mean,target_class,...,quarter,pressure_msl_mean_quarter,pressure_msl_min_quarter,pressure_msl_max_quarter,pressure_msl_std_quarter,surface_pressure_mean_quarter,surface_pressure_min_quarter,surface_pressure_max_quarter,surface_pressure_std_quarter,n_obs_pressure_quarter
0,2009.0,2009-Q3,2,1.0,1.0,0.0,0.0,0.000000,NaN,low,...,3,1016.347826,1009.3,1023.1,3.078657,925.219565,916.7,930.6,2.812657,92
1,2009.0,2009-Q4,1,1.0,0.0,1.0,0.0,1.000000,1.0,medium,...,4,1016.265217,990.5,1031.3,7.303385,920.971739,897.5,933.8,7.262705,92
2,2010.0,2010-Q1,1,1.0,1.0,0.0,0.0,0.000000,NaN,low,...,1,1014.594444,992.6,1026.5,7.834415,917.790000,897.3,929.5,7.288348,90
3,2010.0,2010-Q2,2,1.0,0.0,1.0,0.0,0.500000,1.0,medium,...,2,1015.339560,1002.6,1025.2,5.070719,921.686813,909.5,931.1,5.109908,91
4,2011.0,2011-Q1,1,1.0,0.0,1.0,0.0,1.000000,1.0,medium,...,1,1020.051111,1000.8,1034.0,7.409184,923.067778,904.6,935.4,6.919129,90
5,2011.0,2011-Q3,3,1.0,0.0,1.0,0.0,0.666667,1.0,medium,...,3,1015.238043,1008.0,1024.3,3.562498,924.059783,918.0,931.7,3.047826,92
6,2011.0,2011-Q4,2,1.0,0.5,0.5,0.0,0.500000,1.0,low,...,4,1021.672826,998.5,1035.2,7.538635,925.838043,904.5,935.5,6.467431,92
7,2012.0,2012-Q2,1,1.0,1.0,0.0,0.0,0.000000,NaN,low,...,2,1013.753846,1005.0,1024.4,4.471348,920.707692,910.4,932.2,5.305034,91
8,2012.0,2012-Q3,2,1.0,1.0,0.0,0.0,0.000000,NaN,low,...,3,1015.493478,1006.5,1024.8,3.364678,924.415217,913.4,934.1,3.371305,92
9,2012.0,2012-Q4,1,1.0,0.0,1.0,0.0,0.000000,1.0,medium,...,4,1017.225000,1001.0,1033.0,6.849715,921.461957,906.3,934.1,5.900453,92


## Comprobación final de duplicados

Se valida que el dataset integrado mantiene la propiedad de una fila por trimestre.

In [11]:
print("Duplicados por year_quarter:", df_model.duplicated(subset=["year_quarter"]).sum())

Duplicados por year_quarter: 0


## Revisión de valores nulos

Se revisan los valores nulos del dataset integrado, ya que condicionarán el preprocesado y el modelado posterior.

In [12]:
nulls_df = (
    df_model.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("n_nulls")
    .to_frame()
)

nulls_df[nulls_df["n_nulls"] > 0]

,n_nulls
captures_nonnull_mean,20


## Vista general del dataset final

Se inspeccionan sus columnas, tamaño y primeras filas para confirmar que la tabla final queda lista para modelado.

In [13]:
print("Shape final:", df_model.shape)
print()
print("Columnas:")
print(df_model.columns.tolist())

Shape final: (54, 41)

Columnas:
['year', 'year_quarter', 'n_videos', 'mean_activity_mentions', 'share_low', 'share_medium', 'share_high', 'share_certainty_high', 'captures_nonnull_mean', 'target_class', 'target_class_id', 'quarter_x', 'temp_mean_quarter', 'temp_max_mean_quarter', 'temp_min_mean_quarter', 'precip_sum_quarter', 'rain_sum_quarter', 'snowfall_sum_quarter', 'wind_max_mean_quarter', 'radiation_sum_quarter', 'eto_sum_quarter', 'quarter_y', 'agua_total_mean_quarter', 'agua_total_min_quarter', 'agua_total_max_quarter', 'agua_total_std_quarter', 'agua_actual_mean_quarter', 'agua_actual_min_quarter', 'agua_actual_max_quarter', 'agua_actual_std_quarter', 'n_obs_embalse_quarter', 'quarter', 'pressure_msl_mean_quarter', 'pressure_msl_min_quarter', 'pressure_msl_max_quarter', 'pressure_msl_std_quarter', 'surface_pressure_mean_quarter', 'surface_pressure_min_quarter', 'surface_pressure_max_quarter', 'surface_pressure_std_quarter', 'n_obs_pressure_quarter']


In [14]:
df_model.head(12)

,year,year_quarter,n_videos,mean_activity_mentions,share_low,share_medium,share_high,share_certainty_high,captures_nonnull_mean,target_class,...,quarter,pressure_msl_mean_quarter,pressure_msl_min_quarter,pressure_msl_max_quarter,pressure_msl_std_quarter,surface_pressure_mean_quarter,surface_pressure_min_quarter,surface_pressure_max_quarter,surface_pressure_std_quarter,n_obs_pressure_quarter
0,2009.0,2009-Q3,2,1.000000,1.000000,0.000000,0.0,0.000000,NaN,low,...,3,1016.347826,1009.3,1023.1,3.078657,925.219565,916.7,930.6,2.812657,92
1,2009.0,2009-Q4,1,1.000000,0.000000,1.000000,0.0,1.000000,1.0,medium,...,4,1016.265217,990.5,1031.3,7.303385,920.971739,897.5,933.8,7.262705,92
2,2010.0,2010-Q1,1,1.000000,1.000000,0.000000,0.0,0.000000,NaN,low,...,1,1014.594444,992.6,1026.5,7.834415,917.790000,897.3,929.5,7.288348,90
3,2010.0,2010-Q2,2,1.000000,0.000000,1.000000,0.0,0.500000,1.0,medium,...,2,1015.339560,1002.6,1025.2,5.070719,921.686813,909.5,931.1,5.109908,91
4,2011.0,2011-Q1,1,1.000000,0.000000,1.000000,0.0,1.000000,1.0,medium,...,1,1020.051111,1000.8,1034.0,7.409184,923.067778,904.6,935.4,6.919129,90
5,2011.0,2011-Q3,3,1.000000,0.000000,1.000000,0.0,0.666667,1.0,medium,...,3,1015.238043,1008.0,1024.3,3.562498,924.059783,918.0,931.7,3.047826,92
6,2011.0,2011-Q4,2,1.000000,0.500000,0.500000,0.0,0.500000,1.0,low,...,4,1021.672826,998.5,1035.2,7.538635,925.838043,904.5,935.5,6.467431,92
7,2012.0,2012-Q2,1,1.000000,1.000000,0.000000,0.0,0.000000,NaN,low,...,2,1013.753846,1005.0,1024.4,4.471348,920.707692,910.4,932.2,5.305034,91
8,2012.0,2012-Q3,2,1.000000,1.000000,0.000000,0.0,0.000000,NaN,low,...,3,1015.493478,1006.5,1024.8,3.364678,924.415217,913.4,934.1,3.371305,92
9,2012.0,2012-Q4,1,1.000000,0.000000,1.000000,0.0,0.000000,1.0,medium,...,4,1017.225000,1001.0,1033.0,6.849715,921.461957,906.3,934.1,5.900453,92


## Resumen temporal del dataset

Se comprueba la cobertura temporal final del dataset trimestral que se usará en el notebook de clasificación.

In [15]:
print("Primer trimestre:", df_model["year_quarter"].min())
print("Último trimestre:", df_model["year_quarter"].max())
print("Número total de trimestres:", df_model["year_quarter"].nunique())

Primer trimestre: 2009-Q3
Último trimestre: 2026-Q2
Número total de trimestres: 54


## Guardado del dataset final de modelado

Se persiste la tabla final en formato Parquet y CSV para utilizarla directamente en el notebook 12.

In [16]:
df_model.to_parquet(FINAL_DATASET_PATH, index=False)
df_model.to_csv(FINAL_DATASET_CSV_PATH, index=False, encoding="utf-8")

print("Guardado parquet:", FINAL_DATASET_PATH)
print("Guardado csv:", FINAL_DATASET_CSV_PATH)

Guardado parquet: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity/valmayor_modeling_dataset.parquet
Guardado csv: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity/valmayor_modeling_dataset.csv


## Conclusión

Este notebook deja construido el **dataset final de modelado trimestral** del TFG. A partir del target generado con la extracción LLM y de las variables externas agregadas por trimestre, se obtiene una tabla única, consistente y preparada para el notebook de clasificación.

Con ello queda cerrada la fase de integración de datos del pipeline y se deja listo el siguiente paso: entrenar y evaluar modelos de clasificación sobre los niveles de actividad pesquera.